In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

from Capstone.Rockets.Plots import dots_and_arrows, Interactive_polar, Shape
from Capstone.Rockets.Simulation import run, step
from Capstone.Rockets.Simulation import HScurves, HSintersections
from Capstone.Superformula.Formulas import formula1, formula2
from Capstone.Geometry import cart2pol, pol2cart, rad2deg, normals, magn
from Capstone.utils import describe, clip, shape, pick

## Simulation run
With only the caustics removed.

In [ ]:

d = .05
steps = 2
n = 1000
args = ('trapez wave', 5, 1, -0.66, 0.95, 0, 1, 1)

profile = formula2(*args)

run(profile, d, steps, n)
print("Simulation completed.")

HSdata = HScurves.results()
HSintrsctns = HSintersections.results()

describe(HSdata)
describe(HSintrsctns)




In [ ]:
tmp = HSdata.copy()

tmp.update(HSintrsctns)
# [print(k, v.shape) for k,v in tmp.items() if type(v) not in (int, float, np.float64)]
#tmp = {k: tmp[k] for k in "I, X, Y, Nx, Ny, X1, Y1, Ex, Ey, Px, Py".split(", ")}
describe(tmp)
#[print(k, v.shape) for k,v in tmp.items()]


dots_and_arrows(**tmp, d = d)

## Deriving the algorithm for rarefaction detection
The idea is to look at the segments of the curve and identify which ones are diverging the most from their neighbors. We can do this by looking at the angles between segments, or equivalently the lengths of the segments formed by non-adjacent points. The segments that are diverging the most will have the longest lengths. We can then take the top 2-5% of these segments as indicators of rarefactions, and add a point in between the endpoints of these segments to smooth out the curve.

In [ ]:
I, X,Y, Nx, Ny, Ex, Ey, *_ = tmp.values()

In [ ]:
X

We'll take the results of last step from previous simulation to serve as "clay data" for the observation - idea - implementation - visualization cycle. 


In [ ]:
I = np.asarray(I).ravel()[1000:]
X = np.asarray(X).ravel()[1000:]
Y = np.asarray(Y).ravel()[1000:]
Ex = np.asarray(Ex).ravel()[1000:]
Ey = np.asarray(Ey).ravel()[1000:]
XY = np.stack((X,Y), axis=1)
XY

In [ ]:
XY.shape

Produce segments of the curve:  
- I - index of segments  
- X,Y - starting point of segments  
- c1 - end point of segments  
- v - direction vector of segments (c1 - (X,Y))  


In [ ]:
    
n = XY.shape[0]

c = XY # segment start points

c1 = np.concatenate((c[-1:,:],c[:-1]), axis=0) # segment end points 
v = c1 - c # segment direction vectors

measure lengthts of the segments 


In [ ]:
m = magn(v[:,0], v[:,1])

In [ ]:
px.line(y = m, render_mode='svg', width=600, height=600)

For every segment, calculate which quantile it's length occupies among the rest of segments.  
This gives us a measure of how "divergent" each segment is compared to the rest.  
The segments with the highest quantiles are the ones that are diverging the most from their  
neighbors, which are likely to be the rarefactions we want to identify and smooth out.


In [ ]:

# Count how many elements are strictly less than each element, then normalize
quantiles = np.sum(m[:, None] > m, axis=1) / (len(m) - 1)


In [ ]:
# Select 2% highest 

highs = quantiles > .98


In [ ]:
px.histogram(x = m, color = highs,  width=600, height=600)

In [ ]:
data = {'I': I, 'X': X, 'Y': Y, 'm': m, 'high': highs}

describe(data)

In [ ]:
df = pd.DataFrame(data)
df

Good, rarefaction points identified. Now let's visualize them and see if they make sense.

In [ ]:
px.scatter(df, x='X',y='Y', color='high', hover_data='I', render_mode='svg', width=600, height=600)

Next, we'll add points in between the endpoints of the identified segments to smooth out the curve

To add them, we use np.insert which requires the `index` where to insert, and the `values` to insert  

Small example for illustration:

In [ ]:
x = np.array([0, 1, 2, 3])
xins = np.insert(x, (0,0,2,2), (99,100, 101,102), axis=0)
print(x)
print(xins)


The `index` is the `I` of the identified points +1, since we want to insert after the identified point.

In [ ]:
newI = I[highs]
newI

The `value` is halfway from identified point along it's direction vector.  
This is a primitive interpolation scheme. More sophisticated ones should be used, but this is a good start.

In [ ]:
newXY = XY[highs] + v[highs] / 2

In [ ]:
newX, newY =  newXY[:,0], newXY[:,1]

In [ ]:
newX

In [ ]:
X1 = np.insert(X, newI[:-1], newX[:-1], axis=0)


In [ ]:
Y1 = np.insert(Y, newI[:-1], newY[:-1], axis=0)

In [ ]:
X1.shape, Y1.shape

In [ ]:
isNew = np.zeros_like(X, dtype=bool)
news = np.ones_like(newX, dtype=bool)
isNew = np.insert(isNew, newI[:-1], news[:-1], axis=0)

After insertion, points index needs to be reset.

In [ ]:
I1 = np.arange(X1.shape[0])

In [ ]:
X.shape, Y.shape, XY.shape, newX.shape, newY.shape, newI.shape

In [ ]:
data = {'I': I1, 'X': X1, 'Y': Y1, 'isNew': isNew}
describe(data)

In [ ]:
df = pd.DataFrame(data)
df

In [ ]:
px.scatter(df, x=X1,y=Y1, color=isNew, render_mode='svg', width=600, height=600)

Zooming into a corner of the shape, we can see the added points (blue) and how they smooth out the curve.
Notice however that the sharp curves appear "cut". This is due to that primitive interpolation scheme.  

The fully implemented algorithm is in `rarefactions` function in [Simulation.py](Simulation.py) file.